In [ ]:
from sagemaker.compute_resource_requirements.resource_requirements import ResourceRequirements
from sagemaker.predictor import Predictor
from sagemaker.enums import EndpointType
from sagemaker.model import Model
from sagemaker.session import Session
import boto3

import os
import pandas as pd
import jsonlines
import json
from datetime import datetime

sage_maker_arn = os.getenv('sage_maker_arn')

In [ ]:
## Input model location

# The name of the S3 bucket where your models are stored - replace as necessary (look in S3) :
s3_bucket = "***"

# The directory within your S3 bucket your model is stored in - replace as necessary (look in S3):
bucket_prefix = "***"

# The file name of your model artifact- replace as necessary:
model_filename = "model.safetensors"

# Relative S3 path:
model_s3_key = f"{bucket_prefix}/{model_filename}"

# Combine bucket name, model file name, and relate S3 path to create S3 model URL:
model_url = f"s3://{s3_bucket}/{model_s3_key}"


Define resouce requirements

In [ ]:
resources = ResourceRequirements(
    requests = {
        "num_cpus": 2,  # Number of CPU cores required:
        "num_accelerators": 1, # Number of accelerators required
        "memory": 8192,  # Minimum memory required in Mb (required)
        "copies": 1,
    },
    limits = {},
)


Define predictor deployment settings

In [ ]:

now = datetime.now()
dt_string = now.strftime("%d-%m-%Y-%H-%M-%S")
model_name = "my-sm-model"+dt_string

# build your model with Model class
model = Model(
    name = model_name,
    image_uri = "image-uri", # Docker image
    model_data = model_url,
    role = sage_maker_arn,
    resources = resources,
    predictor_cls = Predictor,
)

# Deploy your model
predictor = model.deploy(
    initial_instance_count = 1,
    instance_type = "ml.g6.xlarge", 
    endpoint_type = EndpointType.INFERENCE_COMPONENT_BASED,
    resources = resources,


Deploy base predictor for comparison

In [ ]:
model_id, model_version = "meta-textgeneration-llama-3-2-1b-instruct", "*"
pretrained_model = JumpStartModel(model_id=model_id, model_version=model_version)
pretrained_predictor = pretrained_model.deploy(accept_eula=accept_eula)

If the models are already deployed, you can regenerate the predictor objects as follows

In [ ]:
# Step 1 - recover the endpoint name

# Option 1
# Find endpoint name in Sagemaker UI 

# Option 2
# using the list_endpoints API
# client = boto3.client('sagemaker')
# print(client.list_endpoints()) # Be sure to differentiate base and finetuned models

In [ ]:
# Step 2 - retrieve defaults

# from sagemaker.predictor import retrieve_default

# ft_endpoint_name = "" ## Choose the desired endpoint from the list
# base_endpoint_name = "" ## Choose the desired endpoint from the list
# predictor = retrieve_default(ft_endpoint_name)
# pretrained_predictor = retrieve_default(base_endpoint_name)

Test the base and finetuned model for SQL generation capacity

In [ ]:
########
## Load data

folder = 'sql_data/' ## Update to location of test dataset as required
test_file = 'bird_test_abrev.jsonl'

lines = []

with jsonlines.open(f'{folder}{test_file}', mode='r') as f:
    lines += f.read()

print(len(lines)) # 2,849

## Load template
with open(f'{folder}template.json', "r") as f:
    template = json.loads(f.read()) # json.loads(f)

inputs = [template['prompt'].format(schema = l['schema'], question = l['question']) for l in lines]

In [ ]:
## Set number of queries to test (can increase as desired)
results_to_test = 20

# Store responses
ft_generated_queries = []
base_generated_queries = []

## Loop through test data
for i, q in enumerate(inputs[:results_to_test]):

    payload = {
    "inputs": q,
    "parameters": {
        "max_new_tokens": 1024,
        "top_p": 0.9,
        "temperature": 0.0,
        "return_full_text": False,
    },
}
    try:
        ## Set eula to true otherwise endpoint will reject requests
        ft_response = predictor.predict(payload, custom_attributes="accept_eula=true")
        ft_generated_queries.append(ft_response['generated_text'])

        base_response = pretrained_.predict(payload, custom_attributes="accept_eula=true")
        base_generated_queries.append(base_response['generated_text'])

    except Exception as e:
        print(e)
    
    print(f'Input {i} processed.')

Compile results

In [ ]:
test_df = pd.DataFrame(lines)

queries_df = pd.DataFrame({'input': inputs[:20],
                           'ft_response': ft_generated_queries,
                           'base_response': base_generated_queries})
queries_df['question'] = queries_df['input'].str.split('User question: ', expand=True)[1]
queries_df['ft_response_query'] = queries_df['response'].str.split('### Response:\n', expand=True)[1]
queries_df = queries_df.merge(test_df[['question', 'sql_query']], on = 'question', how = 'left')

# Save results
queries_df.to_csv(f'ft_results/{endpoint_name} bird test results.csv')

Inspect a single response

In [ ]:
queries_df['responses'][0]

Delete predictors once finished

In [ ]:
## Delete predictors when finished to avoid unnecessary costs
predictor.delete_predictor()
pretrained_predictor.delete_predictor()